# 🥈 Silver Layer — Data Cleaning & Conforming
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/bronze/`, applies cleaning & type casting, writes to `medallion/silver/`.

| Bronze Table | Silver Table | Key Cleaning |
|---|---|---|
| `bronze_orders` | `silver_orders` | Parse dates/numerics, deduplicate, normalise categoricals |
| `bronze_products` | `silver_products` | Normalise SKU, strip whitespace, handle nulls |
| `bronze_discounts` | `silver_discounts` | Parse dates, normalise discount types |
| `bronze_sessions` | `silver_sessions` | Parse dates/numerics, normalise referrer |
| `bronze_rc_orders` | `silver_rc_orders` | Parse dates/numerics, normalise status |
| `bronze_rc_items_checkout` | `silver_rc_items_checkout` | Parse numerics, validate SKU references |
| `bronze_rc_items_recurring` | `silver_rc_items_recurring` | Parse numerics, validate SKU references |
| `bronze_rc_reactivated` | `silver_rc_reactivated` | Parse dates, deduplicate |
| `bronze_rc_churned` | `silver_rc_churned` | Parse dates, deduplicate |

## 0. Setup & Paths

In [74]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
BRONZE_DIR = BASE / 'medallion' / 'bronze'
SILVER_DIR = BASE / 'medallion' / 'silver'
SILVER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Bronze : {BRONZE_DIR}")
print(f"Silver : {SILVER_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Bronze : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/bronze
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver


### Helper utilities

In [75]:
def cleaning_report(df_before, df_after, table_name):
    """Print a before/after summary after cleaning a table."""
    rows_dropped = df_before.shape[0] - df_after.shape[0]
    dup_before   = df_before.duplicated().sum()
    null_before  = df_before.isnull().mean().mean()
    null_after   = df_after.isnull().mean().mean()
    print(f"\n{'─'*55}")
    print(f"  {table_name}")
    print(f"{'─'*55}")
    print(f"  Rows        : {df_before.shape[0]:>7,}  →  {df_after.shape[0]:>7,}  (dropped {rows_dropped:,})")
    print(f"  Duplicates  : {dup_before:>7,}  →  0")
    print(f"  Avg null%   : {null_before:>7.1%}  →  {null_after:>7.1%}")
    print(f"  Cols        : {df_before.shape[1]:>7}  →  {df_after.shape[1]:>7}")


def normalise_str_col(series):
    """Strip whitespace and title-case a string column."""
    return series.str.strip().str.title()


def safe_to_numeric(series):
    """Coerce to numeric; non-parseable values become NaN."""
    return pd.to_numeric(series, errors='coerce')


def safe_to_datetime(series, utc=True):
    """Parse datetime strings; non-parseable values become NaT."""
    dt = pd.to_datetime(series, errors='coerce', utc=utc)
    return dt


print("✅ Helpers loaded")

✅ Helpers loaded


---
## 1. Orders (`bronze_orders` → `silver_orders`)

In [76]:
df = pd.read_parquet(BRONZE_DIR / 'bronze_orders.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 153,828 rows × 70 cols


,ID,Name,Tags,Cancelled At,Cancel: Reason,Processed At,Currency,Source,Checkout ID,Weight Total,Price: Total Line Items,Price: Current Subtotal,Price: Subtotal,Price: Total Discount,Price: Current Total Shipping,Price: Total Shipping,Price: Total Refund,Price: Current Total,Price: Total,Payment: Status,Order Fulfillment Status,Purchase Order Number,Customer: ID,Customer: Tags,Customer: Email Marketing Status,...,Line: Product Handle,Line: Title,Line: Name,Line: Variant ID,Line: Variant Title,Line: SKU,Line: Quantity,Line: Price,Line: Discount,Line: Discount Allocation,Line: Discount per Item,Line: Total,Line: Grams,Line: Requires Shipping,Line: Vendor,Line: Gift Card,Line: Variant SKU,Line: Variant Barcode,Line: Variant Weight,Line: Variant Weight Unit,Line: Variant Inventory Qty,Line: Variant Cost,Line: Variant Price,Line: Variant Compare At Price,_source_file
0,4992746586367,LPSG-9541,None,None,None,2020-12-31 03:27:50 +0800,SGD,17346527233,None,0,83.3,83.3,83.3,0,4.9,4.9,None,88.2,88.2,paid,fulfilled,None,6423777902847,"female, RFM Group Ex lover, RFM Score 113",subscribed,...,None,"Better Whey - 500g Pack, Chocolate Dinosaur","Better Whey - 500g Pack, Chocolate Dinosaur",None,None,None,1,22.15,0,0,0,22.15,0,1,None,0,None,None,None,None,None,None,None,None,1_1.orders-2020_20260505.xlsx
1,4992746586367,LPSG-9541,None,None,None,2020-12-31 03:27:50 +0800,SGD,17346527233,None,None,None,None,None,None,None,None,None,None,None,paid,fulfilled,None,6423777902847,"female, RFM Group Ex lover, RFM Score 113",subscribed,...,None,Green Tea Extract Capsules - 60 Capsules,Green Tea Extract Capsules - 60 Capsules,None,None,None,1,13.5,0,0,0,13.5,0,1,None,0,None,None,None,None,None,None,None,None,1_1.orders-2020_20260505.xlsx
2,4992746586367,LPSG-9541,None,None,None,2020-12-31 03:27:50 +0800,SGD,17346527233,None,None,None,None,None,None,None,None,None,None,None,paid,fulfilled,None,6423777902847,"female, RFM Group Ex lover, RFM Score 113",subscribed,...,None,"Better Whey - 500g Pack, Mango","Better Whey - 500g Pack, Mango",None,None,None,1,22.15,0,0,0,22.15,0,1,None,0,None,None,None,None,None,None,None,None,1_1.orders-2020_20260505.xlsx


In [77]:
# 1.1  Drop fully duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows → {len(df):,} remaining")

# 1.2  Extract discount code lookup before filtering line types
discount_lookup = df[df['Line: Type'] == 'Discount'][['ID', 'Line: Name', 'Line: Total']].copy()
discount_lookup.columns = ['order_id', 'discount_code', 'discount_amount']
discount_lookup['discount_amount'] = safe_to_numeric(discount_lookup['discount_amount']).abs()
discount_lookup = discount_lookup.groupby('order_id').agg(
    discount_code  =('discount_code', lambda x: ', '.join(x.dropna().unique())),
    discount_amount=('discount_amount', 'sum')
).reset_index()
discount_lookup.to_parquet(SILVER_DIR / 'silver_order_discounts_lookup.parquet', index=False)
print(f"✅ Saved: silver_order_discounts_lookup.parquet ({len(discount_lookup):,} orders with discounts)")

# 1.3  Keep only analytically relevant line types
# Shipping Line, Transaction, Discount, Refund Shipping carry no product data
keep_types = ['Line Item', 'Fulfillment Line', 'Refund Line']
before = len(df)
df = df[df['Line: Type'].isin(keep_types)].copy()
print(f"Dropped {before - len(df):,} non-product rows → {len(df):,} remaining")
print(f"Line type breakdown:\n{df['Line: Type'].value_counts()}")

# 1.4  Drop columns with no analytical value
drop_cols = [
    'Purchase Order Number',       # not useful for analytics
    'Browser: Ad URL',             # not useful for analytics
    'Line: Variant Compare At Price', # redundant with product master
    'Line: Variant Barcode',       # not useful for analytics
    'Line: Variant Weight',        # not useful for analytics
    'Line: Variant Weight Unit',   # not useful for analytics
    'Line: Gift Card',             # not useful for analytics
    'Line: Grams',                 # not useful for analytics
    'Weight Total',                # not useful for analytics
    'Checkout ID',                 # internal Shopify ID, not needed
    'Browser: User Agent',         # not useful for analytics
    '_source_file'
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 1.5  Normalise Line: Variant SKU — strip whitespace, uppercase
df['Line: Variant SKU'] = df['Line: Variant SKU'].str.strip().str.upper()

# 1.6  Normalise text fields — strip whitespace, lowercase where appropriate
for col in ['Payment: Status', 'Order Fulfillment Status', 'Source',
            'Currency', 'Shipping: Country', 'Shipping: Country Code']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

# 1.7  Normalise UTM fields — lowercase for consistent grouping
for col in ['Browser: UTM Source', 'Browser: UTM Medium',
            'Browser: UTM Campaign', 'Browser: UTM Content']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

# 1.8  Parse date columns
for col in ['Processed At', 'Cancelled At']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

# 1.9  Cast numeric columns
numeric_cols = [
    'Line: Quantity', 'Line: Price', 'Line: Discount',
    'Line: Discount per Item', 'Line: Total',
    'Price: Total Line Items', 'Price: Subtotal', 'Price: Total Discount',
    'Price: Total Shipping', 'Price: Total Refund', 'Price: Total'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# 1.10  Round monetary columns to 2 decimal places
monetary_cols = [
    'Line: Price', 'Line: Discount', 'Line: Discount per Item', 'Line: Total',
    'Price: Total Line Items', 'Price: Subtotal', 'Price: Total Discount',
    'Price: Total Shipping', 'Price: Total Refund', 'Price: Total'
]
for col in monetary_cols:
    if col in df.columns:
        df[col] = df[col].round(2)

silver_orders = df
cleaning_report(df_raw, silver_orders, 'silver_orders')

Dropped 2944 duplicate rows → 150,884 remaining
✅ Saved: silver_order_discounts_lookup.parquet (8,497 orders with discounts)
Dropped 48,423 non-product rows → 102,461 remaining
Line type breakdown:
Line: Type
Line Item           51220
Fulfillment Line    49920
Refund Line          1321
Name: count, dtype: int64

───────────────────────────────────────────────────────
  silver_orders
───────────────────────────────────────────────────────
  Rows        : 153,828  →  102,461  (dropped 51,367)
  Duplicates  :   2,944  →  0
  Avg null%   :   52.7%  →    42.4%
  Cols        :      70  →       58


In [78]:
print(silver_orders.dtypes)
display(silver_orders.head(3))
silver_orders.to_parquet(SILVER_DIR / 'silver_orders.parquet', index=False)
print("✅ Saved: silver_orders.parquet")

ID                                               object
Name                                             object
Tags                                             object
Cancelled At                        datetime64[ns, UTC]
Cancel: Reason                                   object
Processed At                        datetime64[ns, UTC]
Currency                                         object
Source                                           object
Price: Total Line Items                         float64
Price: Current Subtotal                          object
Price: Subtotal                                 float64
Price: Total Discount                           float64
Price: Current Total Shipping                    object
Price: Total Shipping                           float64
Price: Total Refund                             float64
Price: Current Total                             object
Price: Total                                    float64
Payment: Status                                 

,ID,Name,Tags,Cancelled At,Cancel: Reason,Processed At,Currency,Source,Price: Total Line Items,Price: Current Subtotal,Price: Subtotal,Price: Total Discount,Price: Current Total Shipping,Price: Total Shipping,Price: Total Refund,Price: Current Total,Price: Total,Payment: Status,Order Fulfillment Status,Customer: ID,Customer: Tags,Customer: Email Marketing Status,Customer: SMS Marketing Status,Shipping: Zip,Shipping: City,...,Browser: UTM Medium,Browser: UTM Campaign,Browser: UTM Content,Top Row,Line: Type,Line: ID,Line: Product ID,Line: Product Handle,Line: Title,Line: Name,Line: Variant ID,Line: Variant Title,Line: SKU,Line: Quantity,Line: Price,Line: Discount,Line: Discount Allocation,Line: Discount per Item,Line: Total,Line: Requires Shipping,Line: Vendor,Line: Variant SKU,Line: Variant Inventory Qty,Line: Variant Cost,Line: Variant Price
0,4992746586367,LPSG-9541,None,NaT,None,2020-12-30 19:27:50+00:00,sgd,17346527233,83.3,83.3,83.3,0.0,4.9,4.9,NaN,88.2,88.2,paid,fulfilled,6423777902847,"female, RFM Group Ex lover, RFM Score 113",subscribed,None,461055,Singapore,...,None,None,None,1,Line Item,12664675926271,None,None,"Better Whey - 500g Pack, Chocolate Dinosaur","Better Whey - 500g Pack, Chocolate Dinosaur",None,None,None,1.0,22.15,0.0,0,0.0,22.15,1,None,None,None,None,None
1,4992746586367,LPSG-9541,None,NaT,None,2020-12-30 19:27:50+00:00,sgd,17346527233,NaN,None,NaN,NaN,None,NaN,NaN,None,NaN,paid,fulfilled,6423777902847,"female, RFM Group Ex lover, RFM Score 113",subscribed,None,461055,Singapore,...,None,None,None,None,Line Item,12664675959039,None,None,Green Tea Extract Capsules - 60 Capsules,Green Tea Extract Capsules - 60 Capsules,None,None,None,1.0,13.50,0.0,0,0.0,13.50,1,None,None,None,None,None
2,4992746586367,LPSG-9541,None,NaT,None,2020-12-30 19:27:50+00:00,sgd,17346527233,NaN,None,NaN,NaN,None,NaN,NaN,None,NaN,paid,fulfilled,6423777902847,"female, RFM Group Ex lover, RFM Score 113",subscribed,None,461055,Singapore,...,None,None,None,None,Line Item,12664675991807,None,None,"Better Whey - 500g Pack, Mango","Better Whey - 500g Pack, Mango",None,None,None,1.0,22.15,0.0,0,0.0,22.15,1,None,None,None,None,None


✅ Saved: silver_orders.parquet


---
## 2. Products (`bronze_products` → `silver_products`)

In [79]:
df = pd.read_parquet(BRONZE_DIR / 'bronze_products.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 167 rows × 25 cols


,Handle,Title,Body (HTML),Vendor,Variant SKU,Variant Grams,Variant Price,Variant Compare At Price,Variant Barcode,Image Src,Gift Card,Variant Image,Variant Weight Unit,Cost per item,Status,Included / Singapore,Price / Singapore,Compare At Price / Singapore,Included / Hong Kong,Price / Hong Kong,Compare At Price / Hong Kong,Included / Malaysia,Price / Malaysia,Compare At Price / Malaysia,_source_file
0,lean-protein-peach-oolong-pre-order,LEAN PROTEIN PEACH OOLONG (PRE-ORDER),<p>Elevate your fitness journey with our new Lean Protein! Perfect for a hea...,SG,LEAN-POL-1KG-V1,1000,59.9,None,'0724999807825,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/IMG_9244.jpg?v=177065...,False,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/lean-peachoolong-shad...,kg,17.65,active,True,None,None,True,None,None,True,179,None,2_1.products_master_20260505.xlsx
1,lean-protein-peach-oolong-pre-order,None,None,None,None,None,None,None,None,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/lean-peachoolong-shad...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,2_1.products_master_20260505.xlsx
2,lean-protein-peach-oolong-pre-order,None,None,None,None,None,None,None,None,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/IMG_9181_copy.jpg?v=1...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,2_1.products_master_20260505.xlsx


In [80]:
# 2.1  Drop fully empty rows 
df = df.dropna(how='all')

# 2.2 Forward fill 'Title'
ffill_cols = ['Title']
for col in ffill_cols:
    if col in df.columns:
        df[col] = df[col].ffill()

# 2.2 Drop columns with no analytical value
drop_cols = ['Handle', 'Body (HTML)', 'Vendor', 'Variant Compare At Price', 'Image Src', 'Gift Card', 
            'Variant Image', 'Variant Weight Unit', 'Status', 'Included / Singapore', 
            'Price / Singapore', 'Compare At Price / Singapore', 'Included / Hong Kong', 
            'Price / Hong Kong', 'Compare At Price / Hong Kong',
            'Included / Malaysia', 'Price / Malaysia', 'Compare At Price / Malaysia', '_source_file']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 2.3 Drop rows with no unique 'Variant SKU'
before = len(df)
df = df[df['Variant SKU'].notna()].copy()

# 2.4  Strip whitespace & Uppercase column 'Variant SKU'
df['Variant SKU'] = df['Variant SKU'].str.strip().str.upper()

# 2.5  Deduplicate on SKU (safety net for future exports) 
before = len(df)
df = df.drop_duplicates(subset=['Variant SKU'], keep='first')
print(f"Dedupicates removed {before - len(df)} rows → {len(df)} unique SKUs")

# 2.6  Cast numeric pricing columns 
numeric_cols = ['Variant Price', 'Cost per item']
for col in numeric_cols:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# 2.7 Remove ' from 'Variant Barcode'
if 'Variant Barcode' in df.columns:
    df['Variant Barcode'] = df['Variant Barcode'].str.strip().str.lstrip("'")

# 2.8 Normalise 'Title'
if 'Title' in df.columns:
    df['Title'] = df['Title'].str.strip()

silver_products = df
cleaning_report(df_raw, silver_products, 'silver_products')

Dedupicates removed 0 rows → 56 unique SKUs

───────────────────────────────────────────────────────
  silver_products
───────────────────────────────────────────────────────
  Rows        :     167  →       56  (dropped 111)
  Duplicates  :       4  →  0
  Avg null%   :   65.6%  →     5.1%
  Cols        :      25  →        6


In [81]:
display(silver_products.head(56))
silver_products.to_parquet(SILVER_DIR / 'silver_products.parquet', index=False)
print("✅ Saved: silver_products.parquet")

,Title,Variant SKU,Variant Grams,Variant Price,Variant Barcode,Cost per item
0,LEAN PROTEIN PEACH OOLONG (PRE-ORDER),LEAN-POL-1KG-V1,1000,59.9,0724999807825,17.65
3,[CARTON_20PC] Clear Protein 25g Peach (Pack of 5),CLEAR-PEA-25G-CTN20-5PK-V2,0,0.0,None,83.00
4,[CARTON_20PC] Clear Protein 25g Grape (Pack of 5),CLEAR-GRA-25G-CTN20-5PK-V2,0,0.0,None,83.00
5,[BAG_20PC] Clear Protein Peach 25g,CLEAR-PEA-25G-20PK-V2,0,0.0,None,21.00
6,[BAG_20PC] Clear Protein Grape 25g,CLEAR-GRA-25G-20PK-V2,0,0.0,None,21.00
36,[BAG_15PC] Lean Protein Sachet 40g Thai Milk Tea,LEAN-THA-40G-15PK-V1,0,0.0,None,12.45
37,[BAG_15PC] Lean Protein Sachet 40g Taro,LEAN-TAR-40G-15PK-V1,0,0.0,None,12.45
38,FUEL YOUR JOURNEY OVERSIZED SWEATER,ACC-SWEATER-S-V1,500,80.0,0724999809689,19.00
39,FUEL YOUR JOURNEY OVERSIZED SWEATER,ACC-SWEATER-M-V1,500,80.0,None,19.00
40,FUEL YOUR JOURNEY OVERSIZED SWEATER,ACC-SWEATER-L-V1,500,80.0,None,19.00


✅ Saved: silver_products.parquet


---
## 3. Discounts (`bronze_discounts` → `silver_discounts`)

In [82]:
df = pd.read_parquet(BRONZE_DIR /  'bronze_discounts.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 367 rows × 18 cols


,Name,Value,Value Type,Type,Discount Class,Minimum Purchase Requirements,Combines with Order Discounts,Combines with Product Discounts,Combines with Shipping Discounts,Customer Selection,Context,Times Used In Total,Applies Once Per Customer,Usage Limit Per Code,Status,Start,End,_source_file
0,SG40,-40.0,percentage,Amount Off,order,None,None,None,None,all,all,5,None,None,Expired,2022-09-18 14:39:30 +0800,2022-10-13 10:08:02 +0800,3_1.discounts_export_20260505.csv
1,september,-58.0,percentage,Amount Off,order,None,None,None,None,all,all,72,None,None,Expired,2022-09-24 09:30:00 +0800,2022-10-01 12:00:00 +0800,3_1.discounts_export_20260505.csv
2,9f9abe1f833f,-20.0,percentage,Amount Off,order,10.0,None,None,None,all,all,0,None,1.0,Active,2022-09-24 02:06:29 +0800,None,3_1.discounts_export_20260505.csv


In [83]:
# 3.1  Drop fully empty rows
df = df.dropna(how='all')

# 3.2  Keep only required columns
keep_cols = ['Name', 'Value', 'Value Type', 'Times Used In Total', 'Start', 'End']
df = df[[c for c in keep_cols if c in df.columns]]

# 3.3  Deduplicate on Name
before = len(df)
df = df.drop_duplicates(subset=['Name'], keep='first')
print(f"Dedup removed {before - len(df)} rows → {len(df)} unique discount codes")

# 3.4  Normalise Name — strip whitespace, uppercase
df['Name'] = df['Name'].str.strip().str.upper()

# 3.5  Cast Value to numeric and convert negatives to positive
df['Value'] = safe_to_numeric(df['Value']).abs()

# 3.6  Cast Times Used In Total to numeric
df['Times Used In Total'] = safe_to_numeric(df['Times Used In Total'])

# 3.7  Parse datetime columns
df['Start'] = safe_to_datetime(df['Start'])
df['End']   = safe_to_datetime(df['End'])

silver_discounts = df
cleaning_report(df_raw, silver_discounts, 'silver_discounts')

Dedup removed 1 rows → 366 unique discount codes

───────────────────────────────────────────────────────
  silver_discounts
───────────────────────────────────────────────────────
  Rows        :     367  →      366  (dropped 1)
  Duplicates  :       0  →  0
  Avg null%   :   29.5%  →     5.9%
  Cols        :      18  →        6


In [84]:
display(silver_discounts.head(10))
silver_discounts.to_parquet(SILVER_DIR / 'silver_discounts.parquet', index=False)
print("✅ Saved: silver_discounts.parquet")

,Name,Value,Value Type,Times Used In Total,Start,End
0,SG40,40.0,percentage,5,2022-09-18 06:39:30+00:00,2022-10-13 02:08:02+00:00
1,SEPTEMBER,58.0,percentage,72,2022-09-24 01:30:00+00:00,2022-10-01 04:00:00+00:00
2,9F9ABE1F833F,20.0,percentage,0,2022-09-23 18:06:29+00:00,NaT
3,BFTKSEPT,20.0,percentage,0,2022-09-26 05:29:26+00:00,2022-09-30 16:00:29+00:00
4,ACF,20.0,percentage,0,2022-09-26 10:55:50+00:00,2022-12-31 15:59:55+00:00
5,ZIJIA,20.0,percentage,0,2022-09-26 10:57:37+00:00,2022-12-31 15:59:57+00:00
6,AIDID,20.0,percentage,0,2022-09-26 10:58:22+00:00,2022-12-31 15:59:58+00:00
7,SHEILA,20.0,percentage,1,2022-09-26 10:58:53+00:00,2022-12-31 15:59:58+00:00
8,TAMMY,20.0,percentage,0,2022-09-26 10:59:31+00:00,2023-12-31 15:59:59+00:00
9,JANE,20.0,percentage,0,2022-09-26 11:00:04+00:00,2023-12-31 15:59:59+00:00


✅ Saved: silver_discounts.parquet


---
## 4. Sessions / Traffic (`bronze_sessions` → `silver_sessions`)

In [85]:
df = pd.read_parquet(BRONZE_DIR / 'bronze_sessions.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 137,033 rows × 11 cols


,Referrer source,Referrer name,Session city,UTM campaign,UTM medium,UTM source,Landing page path,Landing page URL,Online store visitors,Sessions,_source_file
0,search,google,Singapore,None,None,None,/,https://lushprotein.com/,3680,4408,4_1.Sessions by referrer_20260505.csv
1,direct,None,None,None,None,None,/,https://www.lushprotein.com/,3340,4308,4_1.Sessions by referrer_20260505.csv
2,direct,None,Singapore,None,None,None,/,https://lushprotein.com/,2713,3972,4_1.Sessions by referrer_20260505.csv


In [86]:
# 4.1  Drop fully empty rows
df = df.dropna(how='all')

# 4.2  Drop 'Landing page URL' and 'Landing page Path'
drop_cols = ['Landing page URL', 'Landing page path']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 4.3  Normalise text fields — lowercase for consistent grouping
text_cols = ['Referrer source', 'Referrer name', 'Session city',
             'UTM campaign', 'UTM medium', 'UTM source', 'Landing page path']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

# 4.4  Cast numeric columns
for col in ['Online store visitors', 'Sessions']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

silver_sessions = df
cleaning_report(df_raw, silver_sessions, 'silver_sessions')


───────────────────────────────────────────────────────
  silver_sessions
───────────────────────────────────────────────────────
  Rows        : 137,033  →  137,033  (dropped 0)
  Duplicates  :       0  →  0
  Avg null%   :   13.1%  →    16.0%
  Cols        :      11  →        9


In [87]:
display(silver_sessions.head(3))
silver_sessions.to_parquet(SILVER_DIR / 'silver_sessions.parquet', index=False)
print("✅ Saved: silver_sessions.parquet")

,Referrer source,Referrer name,Session city,UTM campaign,UTM medium,UTM source,Online store visitors,Sessions,_source_file
0,search,google,singapore,None,None,None,3680,4408,4_1.Sessions by referrer_20260505.csv
1,direct,None,None,None,None,None,3340,4308,4_1.Sessions by referrer_20260505.csv
2,direct,None,singapore,None,None,None,2713,3972,4_1.Sessions by referrer_20260505.csv


✅ Saved: silver_sessions.parquet


---
## 5. Recharge Orders (`bronze_rc_orders` → `silver_rc_orders`)

In [88]:
df = pd.read_parquet(BRONZE_DIR / 'bronze_rc_orders.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 1,215 rows × 11 cols


,metric_date,recharge_order_id,shopify_order_id,order_type,order_total,order_gross_revenue,order_tax,order_shipping,order_discounts,customer_id,_source_file
0,2026-04-07,1306407778,6789789155583,recurring,62.1,62.1,0,0,0,237412247,5_1.orders_combined_20260505.xlsx
1,2026-04-07,1306407662,6789789057279,recurring,107.82,107.82,0,0,0,237898931,5_1.orders_combined_20260505.xlsx
2,2026-04-07,1306910933,6790149636351,checkout,128.01,142.23,0,0,-14.22,243992454,5_1.orders_combined_20260505.xlsx


In [89]:
# 5.1  Drop fully empty rows
df = df.dropna(how='all')

# 5.2  Drop order_tax and order_shipping
drop_cols = ['order_tax', 'order_shipping']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 5.3  Cast date column
df['metric_date'] = safe_to_datetime(df['metric_date'])

# 5.4  Normalise order_type — lowercase, strip whitespace
df['order_type'] = df['order_type'].str.strip().str.lower()

# 5.5  Cast numeric columns
numeric_cols = ['order_total', 'order_gross_revenue', 'order_discounts']
for col in numeric_cols:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# 5.6  Convert order_discounts from negative to positive
df['order_discounts'] = df['order_discounts'].abs()

# 5.7  Round all monetary columns to 2 decimal places
monetary_cols = ['order_total', 'order_gross_revenue', 'order_discounts']
for col in monetary_cols:
    if col in df.columns:
        df[col] = df[col].round(2)

silver_recharge_orders = df
cleaning_report(df_raw, silver_recharge_orders, 'silver_recharge_orders')


───────────────────────────────────────────────────────
  silver_recharge_orders
───────────────────────────────────────────────────────
  Rows        :   1,215  →    1,215  (dropped 0)
  Duplicates  :       0  →  0
  Avg null%   :    0.0%  →     0.0%
  Cols        :      11  →        9


In [90]:
display(silver_recharge_orders.head(3))
silver_recharge_orders.to_parquet(SILVER_DIR / 'silver_recharge_orders.parquet', index=False)
print("✅ Saved: silver_recharge_orders.parquet")

,metric_date,recharge_order_id,shopify_order_id,order_type,order_total,order_gross_revenue,order_discounts,customer_id,_source_file
0,2026-04-07 00:00:00+00:00,1306407778,6789789155583,recurring,62.10,62.10,0.00,237412247,5_1.orders_combined_20260505.xlsx
1,2026-04-07 00:00:00+00:00,1306407662,6789789057279,recurring,107.82,107.82,0.00,237898931,5_1.orders_combined_20260505.xlsx
2,2026-04-07 00:00:00+00:00,1306910933,6790149636351,checkout,128.01,142.23,14.22,243992454,5_1.orders_combined_20260505.xlsx


✅ Saved: silver_recharge_orders.parquet


---
## 6. Recharge Checkout Items → `silver_rc_items_checkout`

In [91]:
df = pd.read_parquet(BRONZE_DIR / 'bronze_rc_items_checkout.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 1,094 rows × 15 cols


,metric_date,recharge_order_id,shopify_order_id,product_id,variant_id,product_sku,purchase_type,product_title,variant_title,order_item_quantity,line_item_price,line_item_tax,line_item_discount,customer_id,_source_file
0,2026-04-07,1307273448,6791342620927,7792625680639,46620967305471,ACC-SHK-V2,onetime,LP CLASSIC SHAKER,White,1,9.46,0,-1.89,244069323,5_2.order_items_checkout_20260505.xlsx
1,2026-04-07,1306657882,6789900239103,8326528991487,44602471940351,CLEAR-GRA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / White Grape,1,66.24,0,0,243978193,5_2.order_items_checkout_20260505.xlsx
2,2026-04-07,1307273448,6791342620927,8326528991487,44602471940351,CLEAR-GRA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / White Grape,2,131.68,0,-13.17,244069323,5_2.order_items_checkout_20260505.xlsx


In [92]:
# 6.1  Drop fully empty rows
df = df.dropna(how='all')

# 6.2  Drop line_item_tax
df = df.drop(columns=[c for c in ['line_item_tax'] if c in df.columns])

# 6.3  Flag rows with null product_sku — cannot join to silver_products
if 'product_sku' in df.columns:
    null_sku = df['product_sku'].isna()
    print(f"Rows with null product_sku: {null_sku.sum()} → flagged as 'missing_sku'")
    df.loc[null_sku, 'item_flag'] = 'missing_sku'

# 6.4  Normalise product_sku — strip whitespace, uppercase
df['product_sku'] = df['product_sku'].str.strip().str.upper()

# 6.5  Normalise text fields
for col in ['purchase_type', 'product_title', 'variant_title']:
    if col in df.columns:
        df[col] = df[col].str.strip()

df['purchase_type'] = df['purchase_type'].str.lower()

# 6.6  Parse date column
df['metric_date'] = safe_to_datetime(df['metric_date'])

# 6.7  Cast numeric columns
numeric_cols = ['order_item_quantity', 'line_item_price', 'line_item_discount']
for col in numeric_cols:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# 6.8  Convert line_item_discount from negative to positive
df['line_item_discount'] = df['line_item_discount'].abs()

# 6.9  Round monetary columns to 2 decimal places
for col in ['line_item_price', 'line_item_discount']:
    if col in df.columns:
        df[col] = df[col].round(2)

silver_recharge_order_items = df
cleaning_report(df_raw, silver_recharge_order_items, 'silver_recharge_order_items')

Rows with null product_sku: 182 → flagged as 'missing_sku'

───────────────────────────────────────────────────────
  silver_recharge_order_items
───────────────────────────────────────────────────────
  Rows        :   1,094  →    1,094  (dropped 0)
  Duplicates  :       0  →  0
  Avg null%   :    1.2%  →     6.7%
  Cols        :      15  →       15


In [93]:
display(silver_recharge_order_items.head(3))
silver_recharge_order_items.to_parquet(SILVER_DIR / 'silver_recharge_order_items.parquet', index=False)
print("✅ Saved: silver_recharge_order_items.parquet")

,metric_date,recharge_order_id,shopify_order_id,product_id,variant_id,product_sku,purchase_type,product_title,variant_title,order_item_quantity,line_item_price,line_item_discount,customer_id,_source_file,item_flag
0,2026-04-07 00:00:00+00:00,1307273448,6791342620927,7792625680639,46620967305471,ACC-SHK-V2,onetime,LP CLASSIC SHAKER,White,1,9.46,1.89,244069323,5_2.order_items_checkout_20260505.xlsx,NaN
1,2026-04-07 00:00:00+00:00,1306657882,6789900239103,8326528991487,44602471940351,CLEAR-GRA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / White Grape,1,66.24,0.00,243978193,5_2.order_items_checkout_20260505.xlsx,NaN
2,2026-04-07 00:00:00+00:00,1307273448,6791342620927,8326528991487,44602471940351,CLEAR-GRA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / White Grape,2,131.68,13.17,244069323,5_2.order_items_checkout_20260505.xlsx,NaN


✅ Saved: silver_recharge_order_items.parquet


---
## 7. Recharge Reactivated Subs → `silver_rc_reactivated`

In [94]:
df = pd.read_parquet(BRONZE_DIR / 'bronze_rc_reactivated.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 50 rows × 5 cols


,metric_date,customer_id,first_subscription_activation_date,reactivated_date,_source_file
0,2026-04-05,175122042,2025-01-16,2026-04-05,5_3.subscribers_reactivated_20260505.xlsx
1,2026-04-01,191230863,2025-04-09,2026-04-01,5_3.subscribers_reactivated_20260505.xlsx
2,2026-04-01,230786752,2026-02-02,2026-04-01,5_3.subscribers_reactivated_20260505.xlsx


In [95]:
# 7.1  Drop fully empty rows
df = df.dropna(how='all')

# 7.2  Parse date columns
for col in ['metric_date', 'first_subscription_activation_date', 'reactivated_date']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

silver_recharge_reactivated = df
cleaning_report(df_raw, silver_recharge_reactivated, 'silver_recharge_reactivated')


───────────────────────────────────────────────────────
  silver_recharge_reactivated
───────────────────────────────────────────────────────
  Rows        :      50  →       50  (dropped 0)
  Duplicates  :       0  →  0
  Avg null%   :    0.0%  →     0.0%
  Cols        :       5  →        5


In [96]:
display(silver_recharge_reactivated.head(3))
silver_recharge_reactivated.to_parquet(SILVER_DIR / 'silver_recharge_reactivated.parquet', index=False)
print("✅ Saved: silver_recharge_reactivated.parquet")

,metric_date,customer_id,first_subscription_activation_date,reactivated_date,_source_file
0,2026-04-05 00:00:00+00:00,175122042,2025-01-16 00:00:00+00:00,2026-04-05 00:00:00+00:00,5_3.subscribers_reactivated_20260505.xlsx
1,2026-04-01 00:00:00+00:00,191230863,2025-04-09 00:00:00+00:00,2026-04-01 00:00:00+00:00,5_3.subscribers_reactivated_20260505.xlsx
2,2026-04-01 00:00:00+00:00,230786752,2026-02-02 00:00:00+00:00,2026-04-01 00:00:00+00:00,5_3.subscribers_reactivated_20260505.xlsx


✅ Saved: silver_recharge_reactivated.parquet


---
## 8. Recharge Churned Subs → `silver_rc_churned`

In [97]:
df = pd.read_parquet(BRONZE_DIR / 'bronze_rc_churned.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 526 rows × 13 cols


,metric_date,subscription_id,customer_id,product_id,product_title,variant_id,product_sku,variant_title,subscription_activation_date,subscription_churn_date,churn_type,cancellation_reason,_source_file
0,2026-04-07,767150230,233442228,8326528991487,CLEAR PROTEIN,44602471907583,CLEAR-PEA-500G-V2,1 x 500g Pack / Peach,2026-02-17,2026-04-07,active,This was created by accident,5_4.subscriptions_churned_20260505.xlsx
1,2026-04-07,580604275,175122113,8548958470399,LEAN PROTEIN,45301479702783,LEAN-THA-1KG-V1,1 x 1kg Pack / Thai Milk Tea,2025-01-16,2026-04-07,active,I no longer use this product,5_4.subscriptions_churned_20260505.xlsx
2,2026-04-07,777767913,237898931,8548958470399,LEAN PROTEIN,46371756998911,LEAN-TAR-1KG-V1,1 x 1kg Pack / Taro,2026-03-10,2026-04-07,active,This is too expensive,5_4.subscriptions_churned_20260505.xlsx


In [98]:
# 8.1  Drop fully empty rows
df = df.dropna(how='all')

# 8.2  Flag rows with null product_sku
if 'product_sku' in df.columns:
    null_sku = df['product_sku'].isna()
    print(f"Rows with null product_sku: {null_sku.sum()} → flagged as 'missing_sku'")
    df.loc[null_sku, 'item_flag'] = 'missing_sku'

# 8.3  Normalise product_sku — strip whitespace, uppercase
df['product_sku'] = df['product_sku'].str.strip().str.upper()

# 8.4  Normalise text fields — strip whitespace
for col in ['product_title', 'variant_title', 'churn_type', 'cancellation_reason']:
    if col in df.columns:
        df[col] = df[col].str.strip()

# 8.5  Standardise (Unknown) cancellation_reason to NaN
df['cancellation_reason'] = df['cancellation_reason'].replace('(Unknown)', pd.NA)

# 8.6  Parse date columns
for col in ['metric_date', 'subscription_activation_date', 'subscription_churn_date']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

silver_recharge_churned = df
cleaning_report(df_raw, silver_recharge_churned, 'silver_recharge_churned')

Rows with null product_sku: 82 → flagged as 'missing_sku'

───────────────────────────────────────────────────────
  silver_recharge_churned
───────────────────────────────────────────────────────
  Rows        :     526  →      526  (dropped 0)
  Duplicates  :       0  →  0
  Avg null%   :    2.4%  →     8.4%
  Cols        :      13  →       14


In [99]:
display(silver_recharge_churned.head(3))
silver_recharge_churned.to_parquet(SILVER_DIR / 'silver_recharge_churned.parquet', index=False)
print("✅ Saved: silver_recharge_churned.parquet")

,metric_date,subscription_id,customer_id,product_id,product_title,variant_id,product_sku,variant_title,subscription_activation_date,subscription_churn_date,churn_type,cancellation_reason,_source_file,item_flag
0,2026-04-07 00:00:00+00:00,767150230,233442228,8326528991487,CLEAR PROTEIN,44602471907583,CLEAR-PEA-500G-V2,1 x 500g Pack / Peach,2026-02-17 00:00:00+00:00,2026-04-07 00:00:00+00:00,active,This was created by accident,5_4.subscriptions_churned_20260505.xlsx,NaN
1,2026-04-07 00:00:00+00:00,580604275,175122113,8548958470399,LEAN PROTEIN,45301479702783,LEAN-THA-1KG-V1,1 x 1kg Pack / Thai Milk Tea,2025-01-16 00:00:00+00:00,2026-04-07 00:00:00+00:00,active,I no longer use this product,5_4.subscriptions_churned_20260505.xlsx,NaN
2,2026-04-07 00:00:00+00:00,777767913,237898931,8548958470399,LEAN PROTEIN,46371756998911,LEAN-TAR-1KG-V1,1 x 1kg Pack / Taro,2026-03-10 00:00:00+00:00,2026-04-07 00:00:00+00:00,active,This is too expensive,5_4.subscriptions_churned_20260505.xlsx,NaN


✅ Saved: silver_recharge_churned.parquet


---
## 9. Recharge Recurring Items → `silver_rc_items_recurring`

In [100]:
df = pd.read_parquet(BRONZE_DIR /  'bronze_rc_items_recurring.parquet')
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

Loaded: 650 rows × 15 cols


,metric_date,recharge_order_id,shopify_order_id,product_id,variant_id,product_sku,purchase_type,product_title,variant_title,order_item_quantity,line_item_price,line_item_tax,line_item_discount,customer_id,_source_file
0,2026-04-07,1306407490,6789788991743,8548958470399,46371756998911,LEAN-TAR-1KG-V1,subscription,LEAN PROTEIN,1 x 1kg Pack / Taro,1,56.91,0,0,231695435,5_5.order_items_recurring_20260505.xlsx
1,2026-04-07,1306407605,6789789090047,8326528991487,44602471907583,CLEAR-PEA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / Peach,2,113.02,0,0,237873185,5_5.order_items_recurring_20260505.xlsx
2,2026-04-07,1306407554,6789789024511,7792627024127,46752160481535,CAP-MUL-BOTTLE-V1,subscription,MULTIVITAMIN CAPSULES,None,1,33.07,0,0,175122062,5_5.order_items_recurring_20260505.xlsx


In [101]:
# 9.1  Drop fully empty rows
df = df.dropna(how='all')

# 9.2  Drop line_item_tax — 100% zeros, no analytical value
df = df.drop(columns=[c for c in ['line_item_tax'] if c in df.columns])

# 9.3  Flag rows with null product_sku
# Discontinued/deleted Shopify variants — not present in any source table,
# cannot be recovered. Exclude from SKU-level joins in gold layer.
if 'product_sku' in df.columns:
    null_sku = df['product_sku'].isna()
    print(f"Rows with null product_sku: {null_sku.sum()} (discontinued variants, unrecoverable)")
    df.loc[null_sku, 'item_flag'] = 'missing_sku'

# 9.4  Normalise product_sku — strip whitespace, uppercase
df['product_sku'] = df['product_sku'].str.strip().str.upper()

# 9.5  Normalise text fields
for col in ['purchase_type', 'product_title', 'variant_title']:
    if col in df.columns:
        df[col] = df[col].str.strip()

df['purchase_type'] = df['purchase_type'].str.lower()

# 9.6  Parse date column
df['metric_date'] = safe_to_datetime(df['metric_date'])

# 9.7  Cast numeric columns
for col in ['order_item_quantity', 'line_item_price', 'line_item_discount']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# 9.8  Convert line_item_discount from negative to positive
df['line_item_discount'] = df['line_item_discount'].abs()

# 9.9  Round monetary columns to 2 decimal places
for col in ['line_item_price', 'line_item_discount']:
    if col in df.columns:
        df[col] = df[col].round(2)

silver_recharge_recurring = df
cleaning_report(df_raw, silver_recharge_recurring, 'silver_recharge_recurring')

Rows with null product_sku: 73 (discontinued variants, unrecoverable)

───────────────────────────────────────────────────────
  silver_recharge_recurring
───────────────────────────────────────────────────────
  Rows        :     650  →      650  (dropped 0)
  Duplicates  :       0  →  0
  Avg null%   :    0.9%  →     6.8%
  Cols        :      15  →       15


In [102]:
display(silver_recharge_recurring.head(3))
silver_recharge_recurring.to_parquet(SILVER_DIR / 'silver_recharge_recurring.parquet', index=False)
print("✅ Saved: silver_recharge_recurring.parquet")

,metric_date,recharge_order_id,shopify_order_id,product_id,variant_id,product_sku,purchase_type,product_title,variant_title,order_item_quantity,line_item_price,line_item_discount,customer_id,_source_file,item_flag
0,2026-04-07 00:00:00+00:00,1306407490,6789788991743,8548958470399,46371756998911,LEAN-TAR-1KG-V1,subscription,LEAN PROTEIN,1 x 1kg Pack / Taro,1,56.91,0.0,231695435,5_5.order_items_recurring_20260505.xlsx,NaN
1,2026-04-07 00:00:00+00:00,1306407605,6789789090047,8326528991487,44602471907583,CLEAR-PEA-500G-V2,subscription,CLEAR PROTEIN,1 x 500g Pack / Peach,2,113.02,0.0,237873185,5_5.order_items_recurring_20260505.xlsx,NaN
2,2026-04-07 00:00:00+00:00,1306407554,6789789024511,7792627024127,46752160481535,CAP-MUL-BOTTLE-V1,subscription,MULTIVITAMIN CAPSULES,None,1,33.07,0.0,175122062,5_5.order_items_recurring_20260505.xlsx,NaN


✅ Saved: silver_recharge_recurring.parquet


---
## 10. Cross-table Validation (Data Quality Check)

In [103]:
# Check 1: All SKUs in orders exist in product master
if 'Line: Variant SKU' in silver_orders.columns and 'Variant SKU' in silver_products.columns:
    order_skus   = set(silver_orders['Line: Variant SKU'].dropna().str.strip().str.upper())
    product_skus = set(silver_products['Variant SKU'].dropna())
    unmatched    = order_skus - product_skus
    print(f"SKUs in orders not found in product master: {len(unmatched):,}")
    if unmatched:
        print("  Sample:", list(unmatched)[:10])

# Check 2: Recharge checkout item order IDs exist in rc_orders
if 'recharge_order_id' in silver_recharge_orders.columns and 'recharge_order_id' in silver_recharge_order_items.columns:
    rc_order_ids  = set(silver_recharge_orders['recharge_order_id'].dropna())
    rc_item_ids   = set(silver_recharge_order_items['recharge_order_id'].dropna())
    orphan_items  = rc_item_ids - rc_order_ids
    print(f"Checkout items with no matching RC order: {len(orphan_items):,}")

# Check 3: Recharge recurring item order IDs exist in rc_orders
if 'recharge_order_id' in silver_recharge_orders.columns and 'recharge_order_id' in silver_recharge_recurring.columns:
    rc_recurring_ids = set(silver_recharge_recurring['recharge_order_id'].dropna())
    orphan_recurring = rc_recurring_ids - rc_order_ids
    print(f"Recurring items with no matching RC order: {len(orphan_recurring):,}")

SKUs in orders not found in product master: 0
Checkout items with no matching RC order: 0
Recurring items with no matching RC order: 0


---
## 11. Silver Layer Summary

In [104]:
silver_tables = {
    'silver_orders'              : silver_orders,
    'silver_products'            : silver_products,
    'silver_discounts'           : silver_discounts,
    'silver_sessions'            : silver_sessions,
    'silver_recharge_orders'     : silver_recharge_orders,
    'silver_recharge_order_items': silver_recharge_order_items,
    'silver_recharge_recurring'  : silver_recharge_recurring,
    'silver_recharge_reactivated': silver_recharge_reactivated,
    'silver_recharge_churned'    : silver_recharge_churned,
}

print(f"{'Table':<35} {'Rows':>8}  {'Cols':>6}")
print('─' * 55)
for name, df in silver_tables.items():
    print(f"{name:<35} {df.shape[0]:>8,}  {df.shape[1]:>6}")

print(f"\n✅ All silver tables saved to: {SILVER_DIR}")

Table                                   Rows    Cols
───────────────────────────────────────────────────────
silver_orders                        102,461      58
silver_products                           56       6
silver_discounts                         366       6
silver_sessions                      137,033       9
silver_recharge_orders                 1,215       9
silver_recharge_order_items            1,094      15
silver_recharge_recurring                650      15
silver_recharge_reactivated               50       5
silver_recharge_churned                  526      14

✅ All silver tables saved to: /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
